In [0]:
# Store WeatherAPI.com API key in Databricks Secrets
# This only needs to be run once

API_KEY_VALUE = "d4929bce95654b14b96133340263105"

print("🔑 Storing WeatherAPI.com API key in Databricks Secrets...")
print("   Scope: api-keys")
print("   Key: weatherapi-key")
print()

try:
    dbutils.secrets.put(
        scope="api-keys",
        key="weatherapi-key",
        string_value=API_KEY_VALUE
    )
    print("✅ API key successfully stored!")
    print("\n📝 The key is now accessible via:")
    print("   dbutils.secrets.get(scope='api-keys', key='weatherapi-key')")
    print("\n⚠️ Note: You may see a confirmation dialog - click 'Confirm' to proceed")
except Exception as e:
    print(f"ℹ️ Secret storage initiated: {e}")
    print("\nIf you see a confirmation dialog, click 'Confirm' to store the secret.")
    print("This is a one-time security confirmation from Databricks.")

# WeatherAPI.com NFL Weather Ingestion

Pull current, forecast, and historical weather data for NFL game locations from WeatherAPI.com.

**Features:**
- Free tier: 1 million API calls/month
- Current weather, 3-day forecast, and historical data
- Detailed wind metrics (speed, gusts, direction)
- Temperature, precipitation, visibility
- Stadium/city-level weather data

**Weather Metrics for Fantasy Impact:**
- **Wind Speed** → QB downgrade, deep ball suppression, kicker accuracy
- **Wind Gusts** → Extreme volatility for passing game
- **Wind Direction** → Crosswinds affect kickers most
- **Temperature** → Cold weather (<32°F) → RB boost, passing downgrade
- **Precipitation** → Ball handling issues, run-heavy game scripts

**Resources:**
- Website: https://www.weatherapi.com
- Docs: https://www.weatherapi.com/docs/
- Free tier: 1M calls/month
- Sign up: https://www.weatherapi.com/signup.aspx

In [0]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ============================================================================
# CONFIGURATION MODE
# ============================================================================
# Set to True to fetch ALL historical weather (2016-2025 game dates)
# Set to False to fetch only UPCOMING games (forecast mode)
HISTORICAL_MODE = True
# ============================================================================

# WeatherAPI.com configuration
BASE_URL = "https://api.weatherapi.com/v1"

# Get API key from Databricks Secrets (or use direct key for this run)
try:
    API_KEY = dbutils.secrets.get(scope="api-keys", key="weatherapi-key")
    print("✅ API key loaded from secrets\n")
except:
    # Use direct API key for this run
    # TODO: Store in secrets via Databricks CLI after this run:
    #   databricks secrets create-scope api-keys
    #   databricks secrets put-secret api-keys weatherapi-key
    API_KEY = "d4929bce95654b14b96133340263105"
    print("✅ API key loaded from configuration\n")
    print("💡 Tip: For production, store this in Databricks Secrets:")
    print("   1. Run: databricks secrets create-scope api-keys")
    print("   2. Run: databricks secrets put-secret api-keys weatherapi-key")
    print()

if HISTORICAL_MODE:
    print("🔄 HISTORICAL MODE: Will fetch weather for all games 2016-2025")
    print("   Using /history.json endpoint for past game dates")
    print("   Joining with bronze_nfl_games for game dates/locations")
else:
    print("⚡ LATEST MODE: Will fetch forecast for upcoming games only")
    print("   Using /forecast.json endpoint (3-day forecast)")
    print("   Joining with bronze_nfl_games for upcoming games")

In [0]:
# NFL stadium locations with historical franchise relocation support
# WeatherAPI.com accepts city names, zip codes, or lat/long

# Current NFL stadiums (2020-2025)
NFL_STADIUMS_CURRENT = {
    # Team: (City, Stadium Name, Is Dome/Retractable)
    'ARI': ('Glendale, AZ', 'State Farm Stadium', True),
    'ATL': ('Atlanta, GA', 'Mercedes-Benz Stadium', True),
    'BAL': ('Baltimore, MD', 'M&T Bank Stadium', False),
    'BUF': ('Orchard Park, NY', 'Highmark Stadium', False),
    'CAR': ('Charlotte, NC', 'Bank of America Stadium', False),
    'CHI': ('Chicago, IL', 'Soldier Field', False),
    'CIN': ('Cincinnati, OH', 'Paycor Stadium', False),
    'CLE': ('Cleveland, OH', 'Cleveland Browns Stadium', False),
    'DAL': ('Arlington, TX', 'AT&T Stadium', True),
    'DEN': ('Denver, CO', 'Empower Field at Mile High', False),
    'DET': ('Detroit, MI', 'Ford Field', True),
    'GB': ('Green Bay, WI', 'Lambeau Field', False),
    'HOU': ('Houston, TX', 'NRG Stadium', True),
    'IND': ('Indianapolis, IN', 'Lucas Oil Stadium', True),
    'JAX': ('Jacksonville, FL', 'TIAA Bank Field', False),
    'KC': ('Kansas City, MO', 'GEHA Field at Arrowhead Stadium', False),
    'LAC': ('Inglewood, CA', 'SoFi Stadium', False),  # 2020+
    'LAR': ('Inglewood, CA', 'SoFi Stadium', False),  # 2020+
    'LV': ('Las Vegas, NV', 'Allegiant Stadium', True),  # 2020+
    'MIA': ('Miami Gardens, FL', 'Hard Rock Stadium', False),
    'MIN': ('Minneapolis, MN', 'U.S. Bank Stadium', True),
    'NE': ('Foxborough, MA', 'Gillette Stadium', False),
    'NO': ('New Orleans, LA', 'Caesars Superdome', True),
    'NYG': ('East Rutherford, NJ', 'MetLife Stadium', False),
    'NYJ': ('East Rutherford, NJ', 'MetLife Stadium', False),
    'PHI': ('Philadelphia, PA', 'Lincoln Financial Field', False),
    'PIT': ('Pittsburgh, PA', 'Acrisure Stadium', False),
    'SEA': ('Seattle, WA', 'Lumen Field', False),
    'SF': ('Santa Clara, CA', "Levi's Stadium", False),
    'TB': ('Tampa, FL', 'Raymond James Stadium', False),
    'TEN': ('Nashville, TN', 'Nissan Stadium', False),
    'WAS': ('Landover, MD', 'FedExField', False)
}

# Historical stadium mappings for franchise relocations (2016-2019)
NFL_STADIUMS_HISTORICAL = {
    # Oakland Raiders → Las Vegas Raiders (moved 2020)
    'OAK': ('Oakland, CA', 'Oakland Coliseum', False),  # 2016-2019
    
    # San Diego Chargers → Los Angeles Chargers
    'SD': ('San Diego, CA', 'Qualcomm Stadium', False),  # 2016 only
    'LAC_2017': ('Carson, CA', 'Dignity Health Sports Park', False),  # 2017-2019
    
    # Los Angeles Rams (returned from St. Louis 2016, moved to SoFi 2020)
    'LA': ('Los Angeles, CA', 'Los Angeles Memorial Coliseum', False),  # 2016-2019
    'LAR_2016': ('Los Angeles, CA', 'Los Angeles Memorial Coliseum', False),  # 2016-2019
}

def get_stadium_for_season(team_code, season):
    """
    Get the correct stadium for a team based on the season.
    Handles franchise relocations and stadium changes.
    
    Args:
        team_code: NFL team abbreviation (e.g., 'OAK', 'LAC', 'LAR')
        season: NFL season year (e.g., 2016, 2020)
    
    Returns:
        Tuple: (city, stadium_name, is_dome)
    """
    # Handle Oakland Raiders → Las Vegas Raiders (2020)
    if team_code == 'OAK':
        if season <= 2019:
            return NFL_STADIUMS_HISTORICAL['OAK']
        else:
            return NFL_STADIUMS_CURRENT['LV']
    
    # Handle Las Vegas Raiders (2020+)
    if team_code == 'LV':
        if season >= 2020:
            return NFL_STADIUMS_CURRENT['LV']
        else:
            # Before 2020, they were Oakland
            return NFL_STADIUMS_HISTORICAL['OAK']
    
    # Handle San Diego Chargers → Los Angeles Chargers
    if team_code == 'SD':
        # SD only valid for 2016
        return NFL_STADIUMS_HISTORICAL['SD']
    
    if team_code == 'LAC':
        if season == 2016:
            # They were still in San Diego
            return NFL_STADIUMS_HISTORICAL['SD']
        elif season <= 2019:
            # 2017-2019: Carson, CA (temporary stadium)
            return NFL_STADIUMS_HISTORICAL['LAC_2017']
        else:
            # 2020+: SoFi Stadium
            return NFL_STADIUMS_CURRENT['LAC']
    
    # Handle Los Angeles Rams (Coliseum → SoFi)
    if team_code in ('LA', 'LAR'):
        if season <= 2019:
            # 2016-2019: LA Coliseum
            return NFL_STADIUMS_HISTORICAL['LAR_2016']
        else:
            # 2020+: SoFi Stadium
            return NFL_STADIUMS_CURRENT['LAR']
    
    # All other teams - return current stadium
    if team_code in NFL_STADIUMS_CURRENT:
        return NFL_STADIUMS_CURRENT[team_code]
    
    # Unknown team
    raise ValueError(f"Unknown team code: {team_code} for season {season}")

print(f"✅ Loaded {len(NFL_STADIUMS_CURRENT)} current NFL stadiums")
print(f"✅ Loaded {len(NFL_STADIUMS_HISTORICAL)} historical stadium mappings")
print(f"\n📍 Franchise Relocations Covered:")
print("   OAK → LV (2020): Oakland Coliseum → Allegiant Stadium")
print("   SD → LAC (2017): Qualcomm → Dignity Health → SoFi (2020)")
print("   LA/LAR (2020): LA Coliseum → SoFi Stadium")
print(f"\n🏟️ Domed/Retractable stadiums (weather neutral): {sum(1 for _, _, is_dome in NFL_STADIUMS_CURRENT.values() if is_dome)}")
print(f"🌤️ Outdoor stadiums: {sum(1 for _, _, is_dome in NFL_STADIUMS_CURRENT.values() if not is_dome)}")

In [0]:
# Fetch games from bronze_nfl_games to get dates and locations
print("="*80)
print("🏈 Fetching NFL Games from Schedule")
print("="*80)

if HISTORICAL_MODE:
    # Fetch all regular season games from 2016-2025
    games_query = """
    SELECT 
        event_id,
        season,
        week,
        date,
        home_team,
        away_team,
        venue,
        game_type
    FROM main.fantasai.bronze_nfl_games
    WHERE source = 'nflverse'
      AND game_type = 'REG'  -- Regular season only
      AND season >= 2016
      AND season <= 2025
    ORDER BY season, week, date
    """
    print("\n📅 Fetching ALL regular season games (2016-2025)...")
else:
    # Fetch only upcoming games (within next 7 days)
    from datetime import datetime, timedelta
    today = datetime.now().strftime('%Y-%m-%d')
    next_week = (datetime.now() + timedelta(days=7)).strftime('%Y-%m-%d')
    
    games_query = f"""
    SELECT 
        event_id,
        season,
        week,
        date,
        home_team,
        away_team,
        venue,
        game_type
    FROM main.fantasai.bronze_nfl_games
    WHERE source = 'nflverse'
      AND game_type = 'REG'
      AND date >= '{today}'
      AND date <= '{next_week}'
    ORDER BY date
    """
    print(f"\n📅 Fetching upcoming games ({today} to {next_week})...")

games_df = spark.sql(games_query).toPandas()

if len(games_df) > 0:
    print(f"\n✅ Found {len(games_df)} games")
    print(f"   Seasons: {games_df['season'].min()}-{games_df['season'].max()}")
    print(f"   Date range: {games_df['date'].min()} to {games_df['date'].max()}")
    print(f"\n📊 Sample games:")
    display(games_df[['season', 'week', 'date', 'home_team', 'away_team', 'venue']].head(10))
else:
    print("\n⚠️ No games found")
    print("\nTroubleshooting:")
    print("1. Make sure you've run 17_nflverse_schedules_ingestion notebook")
    print("2. Check that bronze_nfl_games table has data with source='nflverse'")
    print("3. Verify game dates are in the expected range")

In [0]:
# Fetch weather for each game date using historical or forecast API
import time

if API_KEY is None:
    print("❌ Cannot fetch weather - API key not configured")
    print("Please set up API key first (see Configuration cell)")
else:
    print("\n" + "="*80)
    print("🌦️ Fetching Weather Data")
    print("="*80)
    
    weather_records = []
    api_calls = 0
    errors = 0
    
    # Process games in batches to show progress
    total_games = len(games_df)
    batch_size = 50
    
    for batch_start in range(0, total_games, batch_size):
        batch_end = min(batch_start + batch_size, total_games)
        batch_games = games_df.iloc[batch_start:batch_end]
        
        print(f"\n📊 Processing games {batch_start+1}-{batch_end} of {total_games}...")
        
        for idx, game in batch_games.iterrows():
            game_date = game['date']
            home_team = game['home_team']
            season = int(game['season'])
            
            # Get stadium location (handles historical relocations)
            try:
                city, stadium, is_dome = get_stadium_for_season(home_team, season)
            except ValueError as e:
                print(f"  ⚠️ {e}")
                errors += 1
                continue
            
            try:
                # Use historical API for past games, forecast for future
                if HISTORICAL_MODE:
                    # Historical weather endpoint
                    url = f"{BASE_URL}/history.json"
                    params = {
                        'key': API_KEY,
                        'q': city,
                        'dt': game_date,  # YYYY-MM-DD format
                        'hour': 13  # Default to 1PM for missing game times
                    }
                else:
                    # Forecast endpoint for upcoming games
                    url = f"{BASE_URL}/forecast.json"
                    params = {
                        'key': API_KEY,
                        'q': city,
                        'dt': game_date,
                        'hour': 13
                    }
                
                response = requests.get(url, params=params, timeout=10)
                response.raise_for_status()
                api_calls += 1
                
                data = response.json()
                
                # Extract weather data
                if HISTORICAL_MODE:
                    # Historical response structure
                    forecast_day = data.get('forecast', {}).get('forecastday', [{}])[0]
                    day_data = forecast_day.get('day', {})
                    hour_data = forecast_day.get('hour', [{}])[0]  # Get first hour as sample
                else:
                    # Forecast response structure
                    forecast_day = data.get('forecast', {}).get('forecastday', [{}])[0]
                    day_data = forecast_day.get('day', {})
                    hour_data = forecast_day.get('hour', [{}])[0]
                
                location = data.get('location', {})
                
                weather_record = {
                    'event_id': game['event_id'],
                    'season': season,
                    'week': int(game['week']),
                    'game_date': game_date,
                    'team': home_team,
                    'city': city,
                    'stadium': stadium,
                    'is_dome': is_dome,
                    # Temperature
                    'temp_f': day_data.get('avgtemp_f', 0),
                    'temp_c': day_data.get('avgtemp_c', 0),
                    # Wind (CRITICAL for QB, WR, TE, K projections)
                    'wind_mph': day_data.get('maxwind_mph', 0),
                    'wind_kph': day_data.get('maxwind_kph', 0),
                    'wind_dir': hour_data.get('wind_dir', ''),
                    # Precipitation
                    'precip_in': day_data.get('totalprecip_in', 0),
                    'precip_mm': day_data.get('totalprecip_mm', 0),
                    'humidity': day_data.get('avghumidity', 0),
                    # Conditions
                    'condition_text': day_data.get('condition', {}).get('text', ''),
                    'condition_code': day_data.get('condition', {}).get('code', 0),
                    # Full JSON
                    'raw_data': json.dumps(data)
                }
                
                weather_records.append(weather_record)
                
                # Rate limiting - be respectful to free tier
                time.sleep(0.1)  # 10 requests/second = well under limit
                
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 400:
                    # Bad request - likely invalid date or location
                    errors += 1
                    continue
                else:
                    print(f"  ❌ HTTP Error {e.response.status_code}: {e}")
                    errors += 1
            except Exception as e:
                print(f"  ❌ Error fetching weather for {home_team} on {game_date}: {str(e)[:60]}")
                errors += 1
        
        # Progress update
        print(f"  ✅ Batch complete: {len(weather_records)} weather records, {api_calls} API calls, {errors} errors")
    
    print("\n" + "="*80)
    print("📊 Weather Fetch Summary")
    print("="*80)
    print(f"  Total games processed: {total_games}")
    print(f"  Weather records fetched: {len(weather_records)}")
    print(f"  API calls made: {api_calls}")
    print(f"  Errors: {errors}")
    print(f"  Success rate: {len(weather_records)/total_games*100:.1f}%")
    
    if len(weather_records) > 0:
        weather_df = pd.DataFrame(weather_records)
        print(f"\n✅ Created DataFrame with {len(weather_df)} weather records")
        print(f"\n🌬️ Wind Impact Summary:")
        print(f"   High wind games (>15mph): {len(weather_df[weather_df['wind_mph'] > 15])}")
        print(f"   Extreme wind games (>20mph): {len(weather_df[weather_df['wind_mph'] > 20])}")
        print(f"\n📊 Sample weather data:")
        display(weather_df[['season', 'week', 'game_date', 'team', 'temp_f', 'wind_mph', 'precip_in', 'condition_text']].head(10))
    else:
        print("\n⚠️ No weather data fetched")
        weather_df = pd.DataFrame()

In [0]:
# Fetch current weather for all NFL stadiums
print("Fetching current weather for NFL stadiums...\n")

weather_data = []

for team, (city, stadium, is_dome) in NFL_STADIUMS.items():
    try:
        # Fetch current weather
        url = f"{BASE_URL}/current.json"
        params = {
            'key': API_KEY,
            'q': city,
            'aqi': 'no'  # Don't need air quality data
        }
        
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        # Extract relevant weather fields
        current = data.get('current', {})
        location = data.get('location', {})
        
        weather_record = {
            'team': team,
            'city': city,
            'stadium': stadium,
            'is_dome': is_dome,
            'timestamp': current.get('last_updated'),
            'temp_f': current.get('temp_f'),
            'temp_c': current.get('temp_c'),
            'feels_like_f': current.get('feelslike_f'),
            'wind_mph': current.get('wind_mph'),
            'wind_kph': current.get('wind_kph'),
            'wind_degree': current.get('wind_degree'),
            'wind_dir': current.get('wind_dir'),
            'gust_mph': current.get('gust_mph'),
            'gust_kph': current.get('gust_kph'),
            'pressure_mb': current.get('pressure_mb'),
            'precip_in': current.get('precip_in'),
            'humidity': current.get('humidity'),
            'cloud': current.get('cloud'),
            'visibility_miles': current.get('vis_miles'),
            'condition_text': current.get('condition', {}).get('text'),
            'condition_code': current.get('condition', {}).get('code')
        }
        
        weather_data.append(weather_record)
        print(f"✓ {team}: {temp_f}°F, Wind {wind_mph}mph {wind_dir}")
        
    except Exception as e:
        print(f"✗ {team}: Error - {e}")

# Convert to DataFrame
if weather_data:
    weather_df = pd.DataFrame(weather_data)
    print(f"\n✓ Fetched weather for {len(weather_df)} stadiums")
    print(f"\nSample data:")
    display(weather_df[['team', 'city', 'temp_f', 'wind_mph', 'gust_mph', 'condition_text']].head(10))
else:
    print("\n⚠️ No weather data fetched")
    weather_df = pd.DataFrame()

In [0]:
# Fetch forecast for upcoming game days
# WeatherAPI.com provides 3-day forecast on free tier

print("\nFetching 3-day forecast for stadiums...\n")

forecast_data = []

for team, (city, stadium, is_dome) in list(NFL_STADIUMS.items())[:5]:  # Sample 5 teams for demo
    try:
        url = f"{BASE_URL}/forecast.json"
        params = {
            'key': API_KEY,
            'q': city,
            'days': 3,  # 3-day forecast (free tier limit)
            'aqi': 'no'
        }
        
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        # Extract forecast days
        forecast_days = data.get('forecast', {}).get('forecastday', [])
        
        for day in forecast_days:
            date = day.get('date')
            day_data = day.get('day', {})
            
            forecast_record = {
                'team': team,
                'city': city,
                'stadium': stadium,
                'is_dome': is_dome,
                'forecast_date': date,
                'max_temp_f': day_data.get('maxtemp_f'),
                'min_temp_f': day_data.get('mintemp_f'),
                'avg_temp_f': day_data.get('avgtemp_f'),
                'max_wind_mph': day_data.get('maxwind_mph'),
                'total_precip_in': day_data.get('totalprecip_in'),
                'avg_humidity': day_data.get('avghumidity'),
                'condition_text': day_data.get('condition', {}).get('text'),
                'chance_of_rain': day_data.get('daily_chance_of_rain'),
                'chance_of_snow': day_data.get('daily_chance_of_snow')
            }
            
            forecast_data.append(forecast_record)
        
        print(f"✓ {team}: {len(forecast_days)} day forecast")
        
    except Exception as e:
        print(f"✗ {team}: Error - {e}")

if forecast_data:
    forecast_df = pd.DataFrame(forecast_data)
    print(f"\n✓ Fetched {len(forecast_df)} forecast records")
    display(forecast_df.head(10))
else:
    print("\n⚠️ No forecast data fetched")
    forecast_df = pd.DataFrame()

In [0]:
# Calculate fantasy impact scores based on weather conditions

if 'weather_df' in locals() and len(weather_df) > 0:
    print("\n" + "="*80)
    print("🎮 Calculating Fantasy Weather Impact Scores")
    print("="*80)
    
    df = weather_df.copy()
    
    # Wind impact (affects passing game and kickers)
    # Outdoor games only - domes always get 0 impact
    df['wind_impact_score'] = df.apply(lambda row: 
        0 if row['is_dome'] else
        1 if row['wind_mph'] < 10 else
        2 if row['wind_mph'] < 15 else
        3 if row['wind_mph'] < 20 else
        4,  # Severe (>20mph)
        axis=1
    )
    
    # Temperature impact (cold affects passing, helps RBs)
    # <32F = Severe, 32-40F = Moderate, 40-50F = Light
    df['cold_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] or row['temp_f'] > 50 else
        1 if row['temp_f'] >= 40 else
        2 if row['temp_f'] >= 32 else
        3,  # Severe (<32F)
        axis=1
    )
    
    # Precipitation impact (affects ball handling, favors run game)
    # >0.5" = Severe, 0.1-0.5" = Moderate
    df['precip_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] or row['precip_in'] < 0.1 else
        1 if row['precip_in'] < 0.3 else
        2 if row['precip_in'] < 0.5 else
        3,  # Severe (>0.5")
        axis=1
    )
    
    # Overall weather impact (max of individual scores)
    df['overall_weather_impact'] = df[['wind_impact_score', 'cold_impact_score', 'precip_impact_score']].max(axis=1)
    
    # Position-specific adjustments (percentage impact on projections)
    # Negative = downgrade, Positive = upgrade
    
    # QB: Hurt by wind, cold, and precipitation
    df['qb_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        -1 * (row['wind_impact_score'] * 5 + row['cold_impact_score'] * 3 + row['precip_impact_score'] * 3),
        axis=1
    )
    
    # RB: Helped by cold/precipitation (run-heavy game scripts)
    df['rb_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        (row['cold_impact_score'] * 3 + row['precip_impact_score'] * 4),
        axis=1
    )
    
    # WR: Hurt by wind (especially deep threats), cold, precipitation
    df['wr_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        -1 * (row['wind_impact_score'] * 6 + row['cold_impact_score'] * 2 + row['precip_impact_score'] * 3),
        axis=1
    )
    
    # K: Hurt significantly by wind (accuracy and range)
    df['k_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        -1 * (row['wind_impact_score'] * 8 + row['cold_impact_score'] * 2),
        axis=1
    )
    
    weather_impact_df = df
    
    print(f"\n✅ Calculated impact scores for {len(weather_impact_df)} weather records")
    
    # Show high-impact games
    high_impact = weather_impact_df[weather_impact_df['overall_weather_impact'] >= 2].sort_values('overall_weather_impact', ascending=False)
    
    if len(high_impact) > 0:
        print(f"\n⚠️ High-Impact Weather Games ({len(high_impact)} games with impact >= 2):")
        print(f"\n📊 Top 10 worst weather games:")
        display(high_impact[['season', 'week', 'game_date', 'team', 'temp_f', 'wind_mph', 'precip_in', 'overall_weather_impact', 'qb_adjustment', 'k_adjustment']].head(10))
    else:
        print("\n✅ No high-impact weather games (all games have favorable conditions)")

else:
    print("\n⚠️ No weather data to calculate impact scores")

In [0]:
# Transform game weather data to Spark DataFrame

if 'weather_impact_df' in locals() and len(weather_impact_df) > 0:
    print("\n" + "="*80)
    print("📦 Transforming to Spark DataFrame")
    print("="*80)
    
    rows = []
    for idx, row in weather_impact_df.iterrows():
        # Convert to Spark Row with game-based schema
        spark_row = Row(
            event_id=str(row['event_id']),
            season=int(row['season']),
            week=int(row['week']),
            game_date=str(row['game_date']),
            team=str(row['team']),
            city=str(row['city']),
            stadium=str(row['stadium']),
            is_dome=bool(row['is_dome']),
            # Temperature
            temp_f=float(row['temp_f']) if pd.notna(row['temp_f']) else None,
            temp_c=float(row['temp_c']) if pd.notna(row['temp_c']) else None,
            # Wind
            wind_mph=float(row['wind_mph']) if pd.notna(row['wind_mph']) else None,
            wind_kph=float(row['wind_kph']) if pd.notna(row['wind_kph']) else None,
            wind_dir=str(row['wind_dir']) if pd.notna(row['wind_dir']) and row['wind_dir'] else None,
            # Precipitation
            precip_in=float(row['precip_in']) if pd.notna(row['precip_in']) else None,
            precip_mm=float(row['precip_mm']) if pd.notna(row['precip_mm']) else None,
            humidity=int(row['humidity']) if pd.notna(row['humidity']) else None,
            # Conditions
            condition_text=str(row['condition_text']) if pd.notna(row['condition_text']) and row['condition_text'] else None,
            condition_code=int(row['condition_code']) if pd.notna(row['condition_code']) else None,
            # Impact scores
            wind_impact_score=int(row['wind_impact_score']),
            cold_impact_score=int(row['cold_impact_score']),
            precip_impact_score=int(row['precip_impact_score']),
            overall_weather_impact=int(row['overall_weather_impact']),
            # Position adjustments
            qb_adjustment=int(row['qb_adjustment']),
            rb_adjustment=int(row['rb_adjustment']),
            wr_adjustment=int(row['wr_adjustment']),
            k_adjustment=int(row['k_adjustment']),
            # Full JSON for reference
            raw_data=str(row['raw_data']) if pd.notna(row['raw_data']) else None
        )
        rows.append(spark_row)
    
    weather_spark_df = spark.createDataFrame(rows)
    print(f"\n✅ Created Spark DataFrame with {weather_spark_df.count()} records")
    print(f"\n📊 Sample transformed data:")
    display(weather_spark_df.select('season', 'week', 'game_date', 'team', 'temp_f', 'wind_mph', 'overall_weather_impact').limit(10))
    
else:
    print("\n⚠️ No weather_impact_df to transform")

In [0]:
# Write game weather data to bronze_nfl_weather table

if 'weather_spark_df' in locals():
    print("\n" + "="*80)
    print("💾 Writing Game Weather Data to bronze_nfl_weather")
    print("="*80)
    
    bronze_weather = weather_spark_df.withColumn("ingested_at", F.current_timestamp())
    bronze_weather = bronze_weather.withColumn("source", F.lit("weatherapi_com"))
    
    # Create temp view for merge
    bronze_weather.createOrReplaceTempView("weatherapi_bronze_updates")
    
    # Create table if not exists
    spark.sql("""
        CREATE TABLE IF NOT EXISTS main.fantasai.bronze_nfl_weather (
            event_id STRING,
            season INT,
            week INT,
            game_date STRING,
            team STRING,
            city STRING,
            stadium STRING,
            is_dome BOOLEAN,
            temp_f DOUBLE,
            temp_c DOUBLE,
            wind_mph DOUBLE,
            wind_kph DOUBLE,
            wind_dir STRING,
            precip_in DOUBLE,
            precip_mm DOUBLE,
            humidity INT,
            condition_text STRING,
            condition_code INT,
            wind_impact_score INT,
            cold_impact_score INT,
            precip_impact_score INT,
            overall_weather_impact INT,
            qb_adjustment INT,
            rb_adjustment INT,
            wr_adjustment INT,
            k_adjustment INT,
            raw_data STRING,
            source STRING,
            ingested_at TIMESTAMP
        )
        USING DELTA
        COMMENT 'Bronze layer: NFL game weather data with fantasy impact scores'
    """)
    
    # Perform MERGE to update existing games and insert new ones
    spark.sql("""
        MERGE INTO main.fantasai.bronze_nfl_weather AS target
        USING weatherapi_bronze_updates AS source
        ON target.event_id = source.event_id AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    
    total_written = bronze_weather.count()
    print(f"\n✅ Merged {total_written} weather records into bronze_nfl_weather")
    
    # Show summary by season
    print("\n📊 Breakdown by Season:")
    bronze_weather.groupBy("season").agg(
        F.count("*").alias("games"),
        F.avg("temp_f").alias("avg_temp_f"),
        F.avg("wind_mph").alias("avg_wind_mph"),
        F.sum(F.when(F.col("overall_weather_impact") >= 2, 1).otherwise(0)).alias("high_impact_games")
    ).orderBy("season").show(20)
    
else:
    print("\n⚠️ No weather data to write")

In [0]:
%sql
-- Verify historical weather data coverage
SELECT 
    source,
    season,
    COUNT(DISTINCT event_id) as games,
    ROUND(AVG(temp_f), 1) as avg_temp_f,
    ROUND(AVG(wind_mph), 1) as avg_wind_mph,
    SUM(CASE WHEN overall_weather_impact >= 2 THEN 1 ELSE 0 END) as high_impact_games,
    MIN(game_date) as first_game,
    MAX(game_date) as last_game
FROM main.fantasai.bronze_nfl_weather
WHERE source = 'weatherapi_com'
GROUP BY source, season
ORDER BY season DESC

## WeatherAPI.com Features

### Available Endpoints (Free Tier)
1. **Current Weather** - `/current.json` - Real-time weather
2. **Forecast** - `/forecast.json` - 3-day forecast (free tier)
3. **Historical** - `/history.json` - Past weather data
4. **Sports** - `/sports.json` - Sports-specific data

### Pricing
- **Free**: 1 million calls/month
- **Pro**: $4/month - 5 million calls/month + 14-day forecast
- **Ultra**: Custom pricing - Historical data + more

### Key Weather Metrics for Fantasy Football

#### Wind Impact
- **0-10 mph**: Minimal impact
- **10-15 mph**: Slight passing game downgrade
- **15-20 mph**: Moderate impact - avoid kickers, downgrade deep threats
- **20+ mph**: Severe impact - major QB/K/WR downgrade, RB boost

#### Wind Gusts
- **Sudden gusts >20mph**: Kicker accuracy plummets
- **Crosswinds**: Worse than headwinds for kickers
- **Check wind direction vs stadium orientation**

#### Cold Weather
- **<32°F (freezing)**: Ball handling issues, RB boost (+5-10%)
- **<20°F (extreme cold)**: Major passing downgrade, heavy run game
- **Wind chill**: Use feels_like_f for player comfort

#### Precipitation
- **Rain**: Fumbles increase, passing accuracy down
- **Snow**: Run-heavy game scripts, low scoring
- **Dome games**: Ignore all weather

### Fantasy Adjustments by Position

**Quarterbacks**
- Wind >15mph: -10 to -20% projection
- Cold <32°F: -5 to -10% projection
- Heavy rain: -10% projection

**Running Backs**
- Cold <32°F: +5 to +10% projection (increased usage)
- Heavy rain: +5% projection
- Wind: Minimal impact

**Wide Receivers (Deep Threats)**
- Wind >15mph: -15 to -25% projection
- Cold <32°F: -5% projection
- Target slot receivers instead

**Kickers**
- Wind >12mph: Downgrade significantly
- Wind >20mph: Avoid if possible
- Gusts >20mph: Major risk
- Dome kickers: Always preferable

### Historical Weather Similarity
Use historical endpoint to find similar weather games:
```python
# Find games with similar weather conditions
# Match: wind speed, temp, precip within ranges
# Use for player performance pattern matching
```

### Best Practices
1. **Check weather day-before games** (Saturday for Sunday games)
2. **Monitor forecast updates** (weather changes quickly)
3. **Prioritize dome games in bad weather weeks**
4. **Stack RBs in cold/windy games**
5. **Avoid kickers and deep WRs in 15+ mph wind**